In [0]:
backend_code = '''
from flask import Flask, request, jsonify
from flask_cors import CORS
import json, math, requests, os, re

app = Flask(__name__)
CORS(app)

FACILITIES = []

def load_facilities():
    global FACILITIES
    try:
        table = spark.table("extracted_facilities").toPandas()
        FACILITIES = table.to_dict("records")
        print(f"Loaded {len(FACILITIES)} facilities")
    except:
        print("Table not ready yet")

DATABRICKS_TOKEN = os.environ.get("DATABRICKS_TOKEN","")
WORKSPACE_URL = os.environ.get("WORKSPACE_URL","")

def call_llm(prompt):
    r = requests.post(
        f"https://{WORKSPACE_URL}/serving-endpoints/databricks-llama-4-maverick/invocations",
        headers={"Authorization": f"Bearer {DATABRICKS_TOKEN}"},
        json={"messages":[{"role":"user","content":prompt}],"max_tokens":600}
    )
    return r.json()["choices"][0]["message"]["content"]

def haversine(lat1,lon1,lat2,lon2):
    R=6371
    p1,p2=math.radians(lat1),math.radians(lat2)
    dp=math.radians(lat2-lat1)
    dl=math.radians(lon2-lon1)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R*2*math.atan2(math.sqrt(a),math.sqrt(1-a))

@app.route("/triage", methods=["POST"])
def triage():
    body = request.json
    symptoms = body.get("symptoms","")
    location = body.get("location","India")
    prompt = f"""
You are AarogyaPath emergency triage for rural India.
Patient location: {location}
Symptoms: {symptoms}
Return ONLY valid JSON no markdown:
{{
  "severity": "EMERGENCY or URGENT or MILD",
  "severity_reason": "one sentence why",
  "action": "what to do RIGHT NOW",
  "facility_type_needed": "ICU or Emergency or GeneralSurgery or Dialysis or Clinic or Homecare",
  "call_ambulance": true or false,
  "home_care_steps": ["step1","step2","step3"],
  "red_flags": ["flag1","flag2"],
  "chain_of_thought": "your reasoning"
}}
"""
    raw = call_llm(prompt)
    raw = raw.strip()
    match = re.search(r'\\{{.*\\}}', raw, re.DOTALL)
    if match:
        raw = match.group()
    return jsonify(json.loads(raw))

@app.route("/facilities", methods=["POST"])
def find_facilities():
    body = request.json
    patient_lat = float(body.get("lat", 20.5))
    patient_lon = float(body.get("lon", 78.9))
    needed_type = body.get("facility_type","Emergency")
    state_filter = body.get("state","")

    scored = []
    for f in FACILITIES:
        try:
            lat = float(f.get("latitude") or 0)
            lon = float(f.get("longitude") or 0)
            if not lat or not lon:
                continue
            if state_filter and str(f.get("state","")).lower() != state_filter.lower():
                continue

            dist = haversine(patient_lat, patient_lon, lat, lon)
            trust = int(f.get("trust_score") or 50)
            reachability = max(0, 100 - dist * 0.5)

            cap_map = {
                "ICU":["has_icu","has_emergency"],
                "Emergency":["has_emergency"],
                "GeneralSurgery":["has_surgery","has_anesthesiologist"],
                "Dialysis":["has_dialysis"],
            }
            required = cap_map.get(needed_type,[])
            cap_score = (sum(1 for k in required if str(f.get(k,"")).lower()=="true") / len(required)*100) if required else 70
            last_mile = (trust*0.45)+(reachability*0.30)+(cap_score*0.25)

            scored.append({
                "facility_name": f.get("facility_name",""),
                "city": f.get("address_city",""),
                "state": f.get("state",""),
                "phone": f.get("phone",""),
                "distance_km": round(dist,1),
                "trust_score": trust,
                "trust_level": f.get("trust_level",""),
                "trust_flags": f.get("trust_flags",[]),
                "last_mile_score": round(last_mile,1),
                "has_emergency": f.get("has_emergency"),
                "has_icu": f.get("has_icu"),
                "has_surgery": f.get("has_surgery"),
                "citation": f.get("citation",""),
                "verdict": "RECOMMENDED" if last_mile>=70 else "USE IF NEEDED" if last_mile>=45 else "AVOID",
                "latitude": lat,
                "longitude": lon
            })
        except:
            continue

    scored.sort(key=lambda x: x["last_mile_score"], reverse=True)
    return jsonify({"facilities": scored[:5]})

@app.route("/deserts", methods=["GET"])
def deserts():
    from collections import defaultdict
    pins = defaultdict(lambda: {"missing":[],"state":"","lat":0,"lon":0})
    for f in FACILITIES:
        if int(f.get("trust_score") or 0) < 45:
            continue
        pin = str(f.get("pincode",""))
        missing = []
        if str(f.get("has_icu","")).lower()!="true": missing.append("ICU")
        if str(f.get("has_emergency","")).lower()!="true": missing.append("Emergency")
        if str(f.get("has_surgery","")).lower()!="true": missing.append("Surgery")
        if str(f.get("has_dialysis","")).lower()!="true": missing.append("Dialysis")
        if len(missing) >= 3:
            pins[pin]["missing"] = missing
            pins[pin]["state"] = f.get("state","")
            pins[pin]["lat"] = float(f.get("latitude") or 0)
            pins[pin]["lon"] = float(f.get("longitude") or 0)

    deserts_list = [
        {"pincode":k,"state":v["state"],
         "missing_capabilities":v["missing"],
         "severity":"CRITICAL" if len(v["missing"])>=4 else "HIGH",
         "lat":v["lat"],"lon":v["lon"]}
        for k,v in pins.items()
    ]
    deserts_list.sort(key=lambda x: len(x["missing_capabilities"]), reverse=True)
    return jsonify({
        "total_desert_pins": len(deserts_list),
        "critical_count": sum(1 for d in deserts_list if d["severity"]=="CRITICAL"),
        "deserts": deserts_list[:100]
    })

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status":"ok","facilities":len(FACILITIES)})

if __name__ == "__main__":
    load_facilities()
    app.run(host="0.0.0.0", port=5000, debug=False)
'''

with open("/tmp/app.py", "w") as f:
    f.write(backend_code)
print("✅ Backend file written to /tmp/app.py")

✅ Backend file written to /tmp/app.py


In [0]:
# Get your Databricks username
username = spark.sql("SELECT current_user()").collect()[0][0]
print(f"Your username: {username}")
print(f"Use this experiment path: /Users/{username}/AarogyaPath")

Your username: ishitajaiswal23@gmail.com
Use this experiment path: /Users/ishitajaiswal23@gmail.com/AarogyaPath


In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import requests, json, mlflow, math
from io import BytesIO
import pandas as pd

# Step 1 — Reload dataset
SHEET_ID = "1ZDuDmoQlyxZIEahDBlrMjf2wiWG7xU81"
df = pd.read_excel(BytesIO(requests.get(f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=xlsx").content))
print(f"✅ {len(df)} rows loaded")

# Step 2 — Rebuild results
results = []
for _, row in df.iterrows():
    t = " ".join([str(row.get(c,"") or "") for c in ["capability","specialties","description","procedure","equipment"]]).lower()
    has_emergency = any(w in t for w in ["emergency","24x7","24/7","always open","casualty"])
    has_icu = any(w in t for w in ["icu","intensive care","critical care","ventilator"])
    has_surgery = any(w in t for w in ["surgery","surgical","operation","theatre","appendectomy"])
    has_dialysis = any(w in t for w in ["dialysis","renal","kidney"])
    has_neonatal = any(w in t for w in ["neonatal","nicu","newborn","maternity"])
    has_oncology = any(w in t for w in ["oncology","cancer","chemotherapy"])
    has_anesthesiologist = any(w in t for w in ["anesthes","anaesth"])
    score, flags = 100, []
    if has_surgery and not has_anesthesiologist:
        score -= 40; flags.append("CRITICAL: Surgery claimed but no Anesthesiologist")
    if has_icu and "bed" not in t:
        score -= 20; flags.append("WARNING: ICU claimed but no beds mentioned")
    if not has_emergency and not has_icu and not has_surgery:
        score -= 15; flags.append("INFO: No high-acuity capabilities found")
    trust_level = "HIGH" if score>=75 else "MEDIUM" if score>=45 else "LOW"
    results.append({
        "facility_name": str(row.get("name","")),
        "address_city": str(row.get("address_city","")),
        "state": str(row.get("address_stateOrRegion","")),
        "pincode": str(row.get("address_zipOrPostcode","")),
        "latitude": float(row.get("latitude",0) or 0),
        "longitude": float(row.get("longitude",0) or 0),
        "phone": str(row.get("officialPhone","") or ""),
        "has_emergency": has_emergency, "has_icu": has_icu,
        "has_surgery": has_surgery, "has_dialysis": has_dialysis,
        "has_neonatal": has_neonatal, "has_oncology": has_oncology,
        "has_anesthesiologist": has_anesthesiologist,
        "trust_score": score, "trust_level": trust_level,
        "trust_flags": flags,
        "citation": str(row.get("capability","") or "")[:200],
    })

print(f"✅ {len(results)} facilities extracted!")
print(f"🚨 Emergency: {sum(1 for r in results if r.get('has_emergency'))}")
print(f"🛏️  ICU: {sum(1 for r in results if r.get('has_icu'))}")
print(f"✅ HIGH trust: {sum(1 for r in results if r.get('trust_level')=='HIGH')}")

# Step 3 — MLflow logging
mlflow.set_experiment("/Users/ishitajaiswal23@gmail.com/AarogyaPath")

with mlflow.start_run(run_name="full_dataset_audit"):
    total = len(results)
    high = sum(1 for r in results if r.get('trust_level')=='HIGH')
    medium = sum(1 for r in results if r.get('trust_level')=='MEDIUM')
    low = sum(1 for r in results if r.get('trust_level')=='LOW')
    contradictions = sum(len(r.get('trust_flags',[])) for r in results)
    desert_count = sum(1 for r in results if not r.get('has_emergency') and not r.get('has_icu'))

    mlflow.log_metric("total_facilities", total)
    mlflow.log_metric("high_trust", high)
    mlflow.log_metric("medium_trust", medium)
    mlflow.log_metric("low_trust", low)
    mlflow.log_metric("contradictions_flagged", contradictions)
    mlflow.log_metric("icu_capable", sum(1 for r in results if r.get('has_icu')))
    mlflow.log_metric("emergency_capable", sum(1 for r in results if r.get('has_emergency')))
    mlflow.log_metric("surgery_capable", sum(1 for r in results if r.get('has_surgery')))
    mlflow.log_metric("medical_deserts", desert_count)
    mlflow.log_metric("trust_rate_pct", round(high/total*100, 1))
    mlflow.log_metric("contradiction_rate_pct", round(contradictions/total*100, 1))

    flagged = [r for r in results if r.get('trust_flags')]
    mlflow.log_text(
        "\n\n".join([f"Facility: {r['facility_name']}\nState: {r['state']}\nTrust: {r['trust_score']}\nFlags: {r['trust_flags']}\nCitation: {r.get('citation','')[:150]}" for r in flagged[:20]]),
        "contradictions.txt"
    )

    desert_states = {}
    for r in results:
        if not r.get('has_emergency') and not r.get('has_icu'):
            s = r.get('state','Unknown')
            desert_states[s] = desert_states.get(s,0) + 1
    mlflow.log_text(
        "\n".join([f"{s}: {c}" for s,c in sorted(desert_states.items(), key=lambda x:x[1], reverse=True)]),
        "desert_zones_by_state.txt"
    )

    print(f"\n✅ MLflow logged! Run ID: {mlflow.active_run().info.run_id}")
    print(f"\n📊 FINAL AUDIT SUMMARY:")
    print(f"Total facilities:        {total:,}")
    print(f"HIGH trust:              {high:,} ({round(high/total*100,1)}%)")
    print(f"MEDIUM trust:            {medium:,}")
    print(f"LOW trust:               {low:,}")
    print(f"Contradictions flagged:  {contradictions:,}")
    print(f"Medical deserts:         {desert_count:,}")
    print(f"Contradiction rate:      {round(contradictions/total*100,1)}%")
    print(f"\n👉 Click 'Experiments' in LEFT sidebar → AarogyaPath → screenshot!")

✅ 10000 rows loaded
✅ 10000 facilities extracted!
🚨 Emergency: 1210
🛏️  ICU: 347
✅ HIGH trust: 7757

✅ MLflow logged! Run ID: d8240949140643ee8110308e7c2baa59

📊 FINAL AUDIT SUMMARY:
Total facilities:        10,000
HIGH trust:              7,757 (77.6%)
MEDIUM trust:            2,121
LOW trust:               122
Contradictions flagged:  9,344
Medical deserts:         8,607
Contradiction rate:      93.4%

👉 Click 'Experiments' in LEFT sidebar → AarogyaPath → screenshot!


In [0]:
from pydantic import BaseModel
from typing import Optional, List

class FacilityCapability(BaseModel):
    facility_name: str
    state: str
    has_icu: Optional[bool]
    has_emergency: Optional[bool]
    has_surgery: Optional[bool]
    has_dialysis: Optional[bool]
    has_anesthesiologist: Optional[bool]
    trust_score: int
    trust_level: str
    trust_flags: List[str]
    citation: str

print("✅ Virtue Foundation Schema defined!")
print("\nValidating top 5 facilities:")
for r in results[:5]:
    try:
        v = FacilityCapability(**{k: r.get(k) for k in FacilityCapability.model_fields})
        print(f"✅ {v.facility_name} | {v.trust_level} | Score: {v.trust_score}")
    except Exception as e:
        print(f"⚠️ {r.get('facility_name')} — {e}")

✅ Virtue Foundation Schema defined!

Validating top 5 facilities:
✅ 1000 Smiles Dental Clinic | MEDIUM | Score: 60
✅ 108 Eye And Heath Centre | MEDIUM | Score: 60
✅ 14 Stars Dental | HIGH | Score: 85
✅ 24x7 Family Clinix | HIGH | Score: 85
✅ 3 E EYE CARE | HIGH | Score: 85


In [0]:
from collections import Counter

state_deserts = Counter(
    r.get('state','Unknown') for r in results
    if not r.get('has_icu') and not r.get('has_emergency')
)

print("╔══════════════════════════════════════════════╗")
print("║    NGO ACTION PRIORITY — AarogyaPath         ║")
print("╚══════════════════════════════════════════════╝")
for i, (state, count) in enumerate(state_deserts.most_common(8), 1):
    total_in_state = sum(1 for r in results if r.get('state')==state)
    pct = round(count/total_in_state*100) if total_in_state else 0
    print(f"{i}. {state:<28} {count:>4} desert zones ({pct}%)")

print(f"\n📋 Total states needing intervention: {len(state_deserts)}")
print(f"📋 Total desert facilities: {sum(state_deserts.values()):,}")
print(f"📋 Recommendation: Prioritize {state_deserts.most_common(1)[0][0]}")

╔══════════════════════════════════════════════╗
║    NGO ACTION PRIORITY — AarogyaPath         ║
╚══════════════════════════════════════════════╝
1. Maharashtra                  1316 desert zones (87%)
2. Uttar Pradesh                 888 desert zones (84%)
3. Gujarat                       698 desert zones (83%)
4. Tamil Nadu                    539 desert zones (86%)
5. Kerala                        524 desert zones (88%)
6. West Bengal                   442 desert zones (92%)
7. Rajasthan                     410 desert zones (83%)
8. Delhi                         397 desert zones (89%)

📋 Total states needing intervention: 174
📋 Total desert facilities: 8,607
📋 Recommendation: Prioritize Maharashtra
